# Data Sources:

Based of the amazing work you did in the Movie Industry you've been recruited to the NBA! You are working as the VP of Analytics that helps support a head scout, Mr. Rooney, for the worst team in the NBA probably the Wizards. Mr. Rooney just heard about Data Science and thinks it can solve all the team's problems!!! He wants you to figure out a way to find players that are high performing but maybe not highly paid that you can steal to get the team to the playoffs! 

In this document you will work through a similar process that we did in class with the NBA data (NBA_Perf_22 and nba_salaries_22), merging them together. This is from 22-23 season, feel free to update to 2023-24 season if you want.

https://www.basketball-reference.com/leagues/NBA_2024_totals.html # reference for performance data

https://www.basketball-reference.com/contracts/players.html # reference for salary data

Details: 

- Determine a way to use clustering to estimate based on performance if 
players are under or over paid, generally. 

- Then select players you believe would be best for your team and explain why. Do so in three categories: 
    * Examples that are not good choices (3 or 4) 
    * Several options that are good choices (3 or 4)
    * Several options that could work, assuming you can't get the players in the good category (3 or 4)

- You will decide the cutoffs for each category, so you should be able to explain why you chose them.

- Provide a well commented and clean report of your findings in a separate notebook that can be presented to Mr. Rooney, keeping in mind he doesn't understand...anything. Include a rationale for variables you included in the model, details on your approach and a overview of the results with supporting visualizations. 


Hints:

- Salary is the variable you are trying to understand 
- When interpreting you might want to use graphs that include variables that are the most correlated with Salary
- You'll need to scale the variables before performing the clustering
- Be specific about why you selected the players that you did, more detail is better
- Use good coding practices, comment heavily, indent, don't use for loops unless totally necessary and create modular sections that align with some outcome. If necessary create more than one script,list/load libraries at the top and don't include libraries that aren't used. 
- Be careful for non-traditional characters in the players names, certain graphs won't work when these characters are included.


# Clustering Lab

### Import Libraries and Load Data

In [10]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
performance = pd.read_csv('../data/NBA_Perf_22.csv',encoding='latin1')
performance["Player"] = performance["Player"].str.replace("?", "ć")
salary = pd.read_csv('../data/nba_salaries_22.csv',encoding='utf-8-sig',engine='python')

### Drop NAs, duplicates, and filter data

In [11]:
# Drop NAs
performance = performance.dropna()

# Drop the columns that are not needed
performance = performance.drop(columns=['Player','Pos','Tm','FGA','3PA','2PA','FTA'])

# Exclude players who don't really play
def filter_played(df):
    df = df[df['MP'] > 10]
    df = df[df['G']>20]
    return df
performance = filter_played(performance)

### Standardize the variables

In [12]:
# Select numeric columns
performance['Ne_TOV']=-performance['TOV']
performance = performance.drop(columns=['TOV'])
# Save the column and the index of the dataframe
col = performance.columns
index = performance.index
# Standardize the data
performance = performance.to_numpy()
performance_mean = np.mean(performance, axis=0)
performance_std = np.std(performance, axis=0)
performance=(performance-performance_mean)/performance_std
performance = pd.DataFrame(performance, columns= col, index=index)

### Run the clustering algo with your best guess for K

In [13]:
# Select Feature Data
featured_columns = ['PTS','eFG%','Ne_TOV']
cluster_per = performance[featured_columns]
# Run the clustering algorithm with my best guess for K=3
kmeans_obj_performance = KMeans(n_clusters=3, random_state=1).fit(cluster_per)

### View the results

In [14]:
print(f'The cluster centers are {kmeans_obj_performance.cluster_centers_}')
print(f'The labels are {kmeans_obj_performance.labels_}')
print(f'The inertia is {kmeans_obj_performance.inertia_}')

The cluster centers are [[-0.614497   -0.78853537  0.43951085]
 [-0.24738138  0.88116026  0.43488852]
 [ 1.45070075 -0.01732517 -1.44195213]]
The labels are [0 1 2 0 1 0 0 1 1 0 0 2 1 2 2 0 1 1 1 0 1 1 0 2 2 1 2 0 2 2 2 2 1 1 0 0 2
 1 1 1 0 0 0 0 0 0 1 1 0 0 1 2 2 0 0 1 0 1 2 0 2 0 0 2 1 0 2 0 0 2 1 1 0 0
 2 1 0 1 0 0 2 0 0 0 0 1 2 1 1 1 2 1 0 0 1 0 1 1 1 1 0 2 1 2 2 2 0 1 2 0 0
 0 1 0 0 2 2 1 0 1 1 1 0 0 2 2 0 0 1 2 1 0 1 0 1 0 1 1 1 0 1 2 0 0 1 1 2 0
 0 2 1 2 2 1 1 0 2 1 0 2 1 2 2 1 1 1 1 0 1 1 2 2 2 0 0 2 2 2 1 1 1 1 1 2 2
 2 1 1 0 2 0 0 2 2 2 2 0 0 0 0 2 1 1 0 1 0 0 1 0 0 1 1 1 0 0 0 0 1 1 0 1 2
 2 0 1 2 0 0 2 2 0 0 1 0 1 0 0 0 2 1 1 0 0 0 0 1 1 0 1 0 0 1 2 0 0 1 0 0 2
 0 0 1 2 2 0 2 1 1 1 1 2 0 1 0 1 0 1 0 1 0 1 1 0 0 1 1 0 1 2 2 2 0 1 1 1 1
 1 0 1 0 0 2 1 0 0 0 0 2 2 1 1 2 0 1 0 2 1 0 1 1 0 0 1 0 1 0 1 0 2 0 0 1 0
 0 0 1 0 0 1 1 1 2 0 0 1 1 1 0 2 2 1 1 2 2 1 2 2 0 1 1 0 2 1 0 0 1 1 1 1 1
 0 1 0 0 1 0 1 0 0 2 2 2 2 2 1 0 0 0 0 2 2 0 0 1 2 2 2 0 0 0 0 1 0 1 0 0 0
 0 0 0 1 2 0 1 2 1

### Create a visualization of the results with 2 or 3 variables that you think will best differentiate the clusters

In [19]:
fig = px.scatter_3d(cluster_per, x="PTS", y="eFG%", z="Ne_TOV", color=kmeans_obj_performance.labels_, title="Points vs. eFG% vs. Turnover for player performance")
fig.show(renderer="vscode")

- I pick PTS as it is the most direct contributor to a player's contribution. 
- I chose eFG% because it takes the positional difference of players into account. Centers tend to have a higher percentage as they attack close to the rim, but not necessarily eFG%, as it is calculated using both 2PT and 3PT shots.
- Ne_TOV, the inverse of the turnover, is also a good indicator of performance. To be a star player, it is very important to control one's turnover.

### Evaluate the quality of the clustering using total variance explained and silhouette scores

In [16]:
#Total variance
X = cluster_per.values  # or clust_performance.to_numpy()

# 1. Total Sum of Squares (TSS)
tss = np.sum((X - np.mean(X, axis=0))**2)

# 2. Between‑cluster SS (BSS) = TSS – WSS (where WSS is inertia_)
wss = kmeans_obj_performance.inertia_
bss = tss - wss

# 3. Proportion of variance explained
var_explained = bss / tss

# 4. Silhouette score (single global value)
sil_score = silhouette_score(X, kmeans_obj_performance.labels_)

# 5. Print nicely
print(f'Total variance explained: {var_explained:.2%}')
print(f'Silhouette score: {sil_score:.3f}')


Total variance explained: 60.94%
Silhouette score: 0.347


### Determine the ideal number of clusters using the elbow method and the silhouette coefficient

In [ ]:
# elbow method
wcss = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=1)
    km.fit(X)
    wcss.append(km.inertia_)
elbow_df = pd.DataFrame({"k": range(1,11), "wcss": wcss})

# Compute silhouette scores for k = 2…10
sil_scores = []
ks = range(2, 11)
for k in ks:
    km = KMeans(n_clusters=k, random_state=1)
    labels = km.fit_predict(X)
    sil_scores.append(silhouette_score(X, labels))
sil_df = pd.DataFrame({"k": list(ks), "silhouette": sil_scores})

# Determine best k by silhouette
best_sil_k = sil_df.loc[sil_df["silhouette"].idxmax(), "k"]
print(f"Best number of clusters by silhouette: {best_sil_k}")

# Plot elbow
fig1 = px.line(
    elbow_df, x="k", y="wcss", 
    title="Elbow Method"
)
fig1.update_layout(xaxis=dict(dtick=1))
fig1.show(renderer="vscode")

# Plot silhouette
fig2 = px.line(
    sil_df, x="k", y="silhouette", 
    title="Silhouette Scores"
)
fig2.update_layout(xaxis=dict(dtick=1))
fig2.show(renderer="vscode")


Best number of clusters by silhouette: 2


On the elbow chart, we can see a sharp drop in WCSS (the within-cluster sum of squares) when we add 1 more cluster to the model from k=1. In general, adding clusters always causes over-fitting, so the goal is to have the least amount of clusters as possible while explaining the variance. In this case, the "elbow" of the chart is at k=2, meaning that when we have 2 clusters, the WCSS is relatively low and the model stays away from over-fitting.

### Use the recommended number of cluster (assuming it's different) to retrain your model and visualize the results

In [ ]:
kmeans_obj_performance_best = KMeans(n_clusters=2, random_state=1).fit(cluster_per)
performance['cluster'] = kmeans_obj_performance_best.labels_
fig = px.scatter_3d(performance, x="eFG%", y="PTS", z="G", color='cluster', title="eFG% vs. PTS vs. Turnover votes for player performance")
fig.show(renderer="vscode")

### Once again evaluate the quality of the clustering using total variance explained and silhouette scores

In [12]:
#Total variance

# Suppose `clust_performance` is your DataFrame of features:
X = clust_performance.values  # or clust_performance.to_numpy()

# 1. Total Sum of Squares (TSS)
tss = np.sum((X - np.mean(X, axis=0))**2)

# 2. Between‑cluster SS (BSS) = TSS – WSS (where WSS is inertia_)
wss = kmeans_obj_performance_best.inertia_
bss = tss - wss

# 3. Proportion of variance explained
var_explained = bss / tss

# 4. Silhouette score (single global value)
sil_score = silhouette_score(X, kmeans_obj_performance_best.labels_)

# 5. Print nicely
print(f'Total variance explained: {var_explained:.2%}')
print(f'Silhouette score: {sil_score:.3f}')


Total variance explained: 27.78%
Silhouette score: 0.278


In [13]:
import unicodedata
import pandas as pd
import plotly.express as px

# 0. CLEAN UP salary BEFORE MERGE
# --------------------------------
# Remove dollar signs, commas, whitespace → float
salary['Salary'] = (
    salary['Salary']
      .str.replace(r'[\$,]', '', regex=True)
      .str.strip()
      .astype(float)
)

# 1. Robust normalize function
def normalize_name(name):
    if pd.isna(name):
        return ""
    name = str(name)
    nfkd = unicodedata.normalize("NFKD", name)
    stripped = "".join(c for c in nfkd if not unicodedata.combining(c))
    return stripped.lower().strip()

# 2. Create normalized‐name columns
performance['Player_norm'] = performance['Player'].map(normalize_name)
salary['Player_norm'] = salary['Player'].map(normalize_name)

# 3. Merge salary into performance
performance = performance.merge(
    salary[['Player_norm','Salary']],
    on='Player_norm',
    how='left'
)

# 4. Fill any missing salaries
median_sal = salary['Salary'].median()
performance['Salary'] = performance['Salary'].fillna(median_sal)

# 5. 3D scatter with marker size ∝ Salary
fig = px.scatter_3d(
    performance,
    x="eFG%",
    y="PTS",
    z="G",
    color="cluster",
    size="Salary",
    size_max=12,
    hover_data=["Player","Salary"],
    title="3D Performance Clusters (marker size ∝ salary)"
)
fig.show(renderer="vscode")


### Use the model to select players for Mr. Rooney to consider

In [ ]:
import numpy as np
from scipy.stats import zscore

# 0. (Re‑)compute a simple performance index as the sum of z‑scores
#    – this puts eFG%, PTS, and G on the same scale and gives you one metric.
perf_cols = ["eFG%", "PTS", "G"]

# compute z‑scores column‑wise, skip NaNs if any
zs = performance[perf_cols].apply(zscore)
zs.columns = [f"z_{c}" for c in perf_cols]

performance = performance.join(zs)
performance["perf_index"] = performance[[f"z_{c}" for c in perf_cols]].sum(axis=1)

# 1. Compute cluster medians for perf_index and Salary
cluster_stats = performance.groupby("cluster").agg(
    med_perf   = ("perf_index", "median"),
    med_salary = ("Salary",      "median")
)

# 2. Join those stats back onto each player
performance = performance.join(cluster_stats, on="cluster")

# 3. Classify pay category:
conds = [
    # under‑paid: you perform ≥ your cluster’s median but cost ≤ its median
    (performance["perf_index"] >= performance["med_perf"]) &
    (performance["Salary"]     <= performance["med_salary"]),

    # over‑paid: you perform ≤ your cluster’s median but cost ≥ its median
    (performance["perf_index"] <= performance["med_perf"]) &
    (performance["Salary"]     >= performance["med_salary"])
]
choices = ["underpaid", "overpaid"]

performance["pay_category"] = np.select(conds, choices, default="fair")

# 4. Quick look at your three groups
for cat in ["underpaid","fair","overpaid"]:
    print(f"\n--- {cat.upper()} (4 examples) ---")
    display(
        performance
          .loc[performance["pay_category"]==cat, 
               ["Player","cluster","perf_index","Salary"]]
          .sort_values(["perf_index","Salary"], ascending=[False, True])
          .head(4)
    )


--- UNDERPAID (4 examples) ---


,Player,cluster,perf_index,Salary
244,Nikola Jokić,1,5.585800,4916160.0
361,Dwight Powell,1,4.136188,11080125.0
473,Robert Williams,1,3.975206,10937502.0
148,Daniel Gafford,1,3.769386,1930681.0



--- FAIR (4 examples) ---


,Player,cluster,perf_index,Salary
11,Giannis Antetokounmpo,1,4.978119,42492492.0
159,Rudy Gobert,1,4.809116,38172414.0
439,Karl-Anthony Towns,1,4.656992,33833400.0
235,LeBron James,1,4.536925,44474988.0



--- OVERPAID (4 examples) ---


,Player,cluster,perf_index,Salary
91,John Collins,1,1.801507,23500000.0
402,Anfernee Simons,1,1.795461,22321429.0
228,Kyrie Irving,1,1.755818,36934550.0
227,Brandon Ingram,1,1.587081,31650600.0


### *Another way to measure a player's performance: PER (Player Efficiency Rating)

In [18]:
# Reload the dataset
performance = pd.read_csv('../data/NBA_Perf_22.csv',encoding='latin1')
performance["Player"] = performance["Player"].str.replace("?", "ć")
# Calculate the PER with the PER formula
performance['PER']= (performance['FG']*85.910+performance['STL']*53.897+performance['3P']*51.757+performance['FT']*46.845+performance['BLK']*39.190+performance['ORB']*39.190+performance['DRB']*14.707+performance['AST']*34.677-(performance['FGA']-performance['FG'])*39.190-(performance['FTA']-performance['FT'])*20.091-performance['TOV']*53.897-performance['PF']*17.174)*(1/performance['MP'])
performance['PER'] = performance['PER'].round(2)
performance['PER'] = performance['PER'].replace([np.inf, -np.inf], np.nan)
performance['PER'] = performance['PER'].fillna(0)
performance['PER'] = performance['PER'].astype(float)
def filter_played(df):
    df = df[df['MP'] > 10]
    df = df[df['G']>20]
    return df
performance = filter_played(performance)
top_10_players = performance.sort_values(by='PER', ascending=False).head(10)
print(top_10_players[['Player', 'PER']])

                    Player    PER
391           Nikola Jokić  38.48
15   Giannis Antetokounmpo  37.84
206            Joel Embiid  35.79
368           LeBron James  30.96
198           Kevin Durant  30.52
808             Trae Young  30.17
526              Ja Morant  29.94
178            Luka Donćić  29.48
736     Karl-Anthony Towns  29.00
160          Anthony Davis  28.17
